# MNIST Classification using MLP with TensorFlow/Keras


In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import numpy as np
import pandas as pd


## Load and preprocess the MNIST dataset

In [2]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalize pixel values to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Flatten 28x28 images into 784-length vectors
X_train = X_train.reshape(-1, 784)
X_test = X_test.reshape(-1, 784)

# One-hot encode labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)


11490434/11490434 [==============================] - 3s 0us/step
Training shape: (60000, 784)
Test shape: (10000, 784)


## Function to build an MLP model

In [4]:
def build_mlp(hidden_layers, activation_function='relu'):
    model = models.Sequential()
    model.add(layers.Input(shape=(784,)))

    for neurons in hidden_layers:
        model.add(layers.Dense(neurons, activation=activation_function))

    model.add(layers.Dense(10, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


## Part (a): Experiment with different hidden layers and neurons

In [5]:
architectures = {
    '1_hidden_128': [128],
    '2_hidden_256_128': [256, 128],
    '3_hidden_512_256_128': [512, 256, 128]
}

architecture_results = []

print('\n===== Part (a): Architecture Comparison =====')

for name, hidden_layers in architectures.items():
    print(f'\nTraining architecture: {name} -> {hidden_layers}')

    model = build_mlp(hidden_layers=hidden_layers, activation_function='relu')

    history = model.fit(
        X_train, y_train_cat,
        epochs=5,
        batch_size=128,
        validation_split=0.1,
        verbose=1
    )

    test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)

    architecture_results.append({
        'Model': name,
        'Hidden Layers': str(hidden_layers),
        'Activation': 'relu',
        'Test Accuracy': test_acc
    })

    print(f'Test Accuracy for {name}: {test_acc:.4f}')



===== Part (a): Architecture Comparison =====

Training architecture: 1_hidden_128 -> [128]


Epoch 1/5


422/422 [==============================] - 2s 2ms/step - loss: 0.3868 - accuracy: 0.8922 - val_loss: 0.1756 - val_accuracy: 0.9542
Epoch 2/5
422/422 [==============================] - 1s 2ms/step - loss: 0.1786 - accuracy: 0.9498 - val_loss: 0.1249 - val_accuracy: 0.9648
Epoch 3/5
422/422 [==============================] - 1s 2ms/step - loss: 0.1296 - accuracy: 0.9624 - val_loss: 0.1057 - val_accuracy: 0.9687
Epoch 4/5
422/422 [==============================] - 1s 2ms/step - loss: 0.1020 - accuracy: 0.9702 - val_loss: 0.0971 - val_accuracy: 0.9718
Epoch 5/5
422/422 [==============================] - 1s 2ms/step - loss: 0.0823 - accuracy: 0.9768 - val_loss: 0.0980 - val_accuracy: 0.9710
Test Accuracy for 1_hidden_128: 0.9683

Training architecture: 2_hidden_256_128 -> [256, 128]
Epoch 1/5
422/422 [==============================] - 2s 3ms/step - loss: 0.2880 - accuracy: 0.9181 - val

## Part (b): Compare ReLU, Sigmoid, and Tanh activation functions

In [6]:
activation_functions = ['relu', 'sigmoid', 'tanh']
activation_results = []

fixed_architecture = [256, 128]

print('\n===== Part (b): Activation Function Comparison =====')

for act in activation_functions:
    print(f'\nTraining with activation: {act}')

    model = build_mlp(hidden_layers=fixed_architecture, activation_function=act)

    history = model.fit(
        X_train, y_train_cat,
        epochs=5,
        batch_size=128,
        validation_split=0.1,
        verbose=1
    )

    test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)

    activation_results.append({
        'Model': f'MLP_{act}',
        'Hidden Layers': str(fixed_architecture),
        'Activation': act,
        'Test Accuracy': test_acc
    })

    print(f'Test Accuracy with {act}: {test_acc:.4f}')



===== Part (b): Activation Function Comparison =====

Training with activation: relu
Epoch 1/5
422/422 [==============================] - 2s 3ms/step - loss: 0.2886 - accuracy: 0.9170 - val_loss: 0.1208 - val_accuracy: 0.9647
Epoch 2/5
422/422 [==============================] - 1s 3ms/step - loss: 0.1116 - accuracy: 0.9660 - val_loss: 0.0936 - val_accuracy: 0.9728
Epoch 3/5
422/422 [==============================] - 1s 3ms/step - loss: 0.0733 - accuracy: 0.9776 - val_loss: 0.0776 - val_accuracy: 0.9782
Epoch 4/5
422/422 [==============================] - 1s 3ms/step - loss: 0.0527 - accuracy: 0.9838 - val_loss: 0.0764 - val_accuracy: 0.9772
Epoch 5/5
422/422 [==============================] - 1s 3ms/step - loss: 0.0406 - accuracy: 0.9871 - val_loss: 0.0749 - val_accuracy: 0.9802
Test Accuracy with relu: 0.9782

Training with activation: sigmoid
Epoch 1/5
422/422 [==============================] - 2s 4ms/step - loss: 0.6465 - accuracy: 0.8393 - val_loss: 0.2369 - val_accuracy: 0.9350
E

## Display results

In [7]:
print('\n===== Architecture Results =====')
architecture_df = pd.DataFrame(architecture_results)
print(architecture_df)

print('\n===== Activation Function Results =====')
activation_df = pd.DataFrame(activation_results)
print(activation_df)



===== Architecture Results =====
                  Model    Hidden Layers Activation  Test Accuracy
0          1_hidden_128            [128]       relu         0.9683
1      2_hidden_256_128       [256, 128]       relu         0.9767
2  3_hidden_512_256_128  [512, 256, 128]       relu         0.9739

===== Activation Function Results =====
         Model Hidden Layers Activation  Test Accuracy
0     MLP_relu    [256, 128]       relu         0.9782
1  MLP_sigmoid    [256, 128]    sigmoid         0.9651
2     MLP_tanh    [256, 128]       tanh         0.9765


## Best models

In [8]:
best_architecture = architecture_df.loc[architecture_df['Test Accuracy'].idxmax()]
best_activation = activation_df.loc[activation_df['Test Accuracy'].idxmax()]

print('\nBest architecture model:')
print(best_architecture)

print('\nBest activation model:')
print(best_activation)



Best architecture model:
Model            2_hidden_256_128
Hidden Layers          [256, 128]
Activation                   relu
Test Accuracy              0.9767
Name: 1, dtype: object

Best activation model:
Model              MLP_relu
Hidden Layers    [256, 128]
Activation             relu
Test Accuracy        0.9782
Name: 0, dtype: object
